# EDA: Spotify Hit Predictor

Этот ноутбук предполагает, что уже создан файл `data/processed/model_table.csv` через `python -m src.make_dataset`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/processed/model_table.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/processed/model_table_sample.csv')

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape, df['target_future_top50'].value_counts(normalize=True)

## Распределение target

In [ ]:
counts = df['target_future_top50'].value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar(['not future hit', 'future hit'], counts.values)
plt.title('Target distribution')
plt.ylabel('Number of tracks')
plt.show()

## Ранняя позиция в чарте и будущий успех

In [ ]:
data = [
    df.loc[df['target_future_top50'] == 0, 'first_rank'].dropna(),
    df.loc[df['target_future_top50'] == 1, 'first_rank'].dropna(),
]
plt.figure(figsize=(6,4))
plt.boxplot(data, labels=['not future hit', 'future hit'])
plt.title('First rank by future success')
plt.ylabel('First rank; lower is better')
plt.show()

## Корреляции с target

In [ ]:
num = df.select_dtypes(include='number')
corr = num.corr(numeric_only=True)['target_future_top50'].drop('target_future_top50').sort_values(key=lambda x: x.abs(), ascending=False)
corr.head(20)

## Audio features

In [ ]:
for col in ['audio_energy', 'audio_danceability', 'audio_valence', 'audio_tempo', 'duration_min']:
    if col not in df.columns:
        continue
    plt.figure(figsize=(6,4))
    df.loc[df['target_future_top50'] == 0, col].dropna().hist(alpha=0.6, bins=30, label='not future hit')
    df.loc[df['target_future_top50'] == 1, col].dropna().hist(alpha=0.6, bins=30, label='future hit')
    plt.title(col)
    plt.legend()
    plt.show()